# NIST StRD: one-way analysis of variance

Does Julia's one-way ANOVA give NIST's certified sums of squares, F statistic, R-squared and residual standard deviation? The 11 ANOVA datasets of NIST's Statistical Reference Datasets, from 25 to 18,009 observations, run through `OneWayANOVATest` of HypothesisTests, the One-Way ANOVA test of the Hypothesis Tests panel.

The Statistical Reference Datasets (StRD) of the US National Institute of Standards and Technology (NIST) were built to test the numerical accuracy of statistical software. Each comes with values certified to 15 significant digits, computed in multiple precision. Some are real data; others are constructed to be hard, with many leading digits that do not change from one observation to the next, so that software which loses precision gives wrong digits. NIST grades each dataset's difficulty as lower, average or higher.

The data files sit beside this notebook as NIST publishes them (https://www.itl.nist.gov/div898/strd/), and every certified value is read from its file. The number of significant digits a value shares with NIST is the log relative error of McCullough (1998, *The American Statistician* 52(4), 358-366), the usual measure for these datasets: -log10(|value - certified| / |certified|), or -log10|value| when the certified value is 0, rounded down and at most 15.

**The data limit.** Julia stores each number of a file as a Float64, a binary number with 15 to 17 significant digits, and storing alone can change a number: `1000000000000.4` is stored as `1000000000000.4000244140625`. The **data limit** repeats each calculation in 256-bit `BigFloat` arithmetic on the stored numbers: no Float64 program can do better on this data. Where Julia falls short of the data limit, its method lost digits; where the data limit falls short of 15, the storing did.

In [ ]:
Pollis.packages("HypothesisTests")

## Set-up

`read_strd` reads a NIST file, `certified` reads a certified value from its header, `digits_same` counts the significant digits a value shares with it, `report` prints a dataset's values below NIST's and `summary` the fewest digits in each dataset.

In [ ]:
using HypothesisTests

using Printf

"""
	read_strd(name)

Read the NIST StRD file `name.dat` in this folder.

Gives: `data`, a matrix with one column per variable, the response first, and `header`, the text
above the data, which holds the certified values.
"""
function read_strd(name)
	lines = readlines(name * ".dat")
	range = match(r"Data\s*:?\s*\(?lines\s+(?<first>\d+)\s+to\s+(?<last>\d+)", join(lines[1:12], "\n"))
	first, last = parse(Int, range[:first]), parse(Int, range[:last])
	data = permutedims(reduce(hcat, [parse.(Float64, split(line)) for line in lines[first:last]]))
	return (data = data, header = join(lines[1:first - 1], "\n"))
end

"""
	certified(header, pattern)

The number captured as `v` by `pattern` in a file's header.
"""
certified(header, pattern) = parse(Float64, match(pattern, header)[:v])

"""
	digits_same(value, certified)

The number of significant digits `value` shares with `certified`: McCullough's log relative
error, or the log absolute error when `certified` is 0, rounded down and at most 15.
"""
function digits_same(value, certified)
	isfinite(value) || return 0
	error = certified == 0 ? abs(value) : abs(value - certified) / abs(certified)
	return error == 0 ? 15 : clamp(floor(Int, -log10(error)), 0, 15)
end

"""
	report(title, columns, rows)

Print each certified value with the values of `columns` below it, and the significant digits
each shares with NIST. Each row is `(label, certified, values...)`, one value for each column.

Gives: the fewest digits shared in each column.
"""
function report(title, columns, rows)
	println("\n", title)
	fewest = fill(15, length(columns))
	for (label, c, values...) in rows
		@printf("%-16s%-12s%25.15g\n", label, "certified", c)
		for (k, (column, v)) in enumerate(zip(columns, values))
			d = digits_same(v, c)
			fewest[k] = min(fewest[k], d)
			@printf("%-16s%-12s%25.15g  %s\n", "", column, v, d == 0 ? "DIFFERENT" : "same to $d significant digits")
		end
	end
	return fewest
end

"""
	summary(columns, results)

Print the fewest significant digits shared with NIST in each dataset, one column for each of
`columns`. Each result is `(name, level, fewest)`.
"""
function summary(columns, results)
	@printf("%-12s%-12s", "dataset", "difficulty")
	foreach(column -> @printf("%14s", column), columns)
	println()
	for (name, level, fewest) in results
		@printf("%-12s%-12s", name, level)
		foreach(d -> @printf("%14s", d == 0 ? "DIFFERENT" : string(d)), fewest)
		println()
	end
end

## 1. The datasets

The 11 datasets, with NIST's difficulty. SiRstv and AtmWtAg are measurements, of silicon resistivity and of the atomic weight of silver. SmLs01 to SmLs09 are constructed: 9 groups of 21, 201 or 2,001 values near 1.4 (SmLs01 to SmLs03), 1000000.4 (SmLs04 to SmLs06) or 1000000000000.4 (SmLs07 to SmLs09).

In [ ]:
datasets = [
	"SiRstv" => "lower", "SmLs01" => "lower", "SmLs02" => "lower", "SmLs03" => "lower",
	"AtmWtAg" => "average", "SmLs04" => "average", "SmLs05" => "average", "SmLs06" => "average",
	"SmLs07" => "higher", "SmLs08" => "higher", "SmLs09" => "higher",
]

## 2. The ANOVA table

Each file has the group in its first column and the response in its second. `OneWayANOVATest` keeps the between-groups and within-groups sums of squares group by group; their totals give the rest of the table. It works in Float64 whatever it is given, so the data limit writes out the textbook formulas in `BigFloat`: the squared deviations of the group means from the grand mean, and of each value from its group mean.

In [ ]:
"""
	anova_table(between, within, df_between, df_within)

The one-way ANOVA table from its two sums of squares and their degrees of freedom.
"""
function anova_table(between, within, df_between, df_within)
	return (between = between, between_mean = between / df_between, within = within, within_mean = within / df_within,
		F = (between / df_between) / (within / df_within), r2 = between / (between + within), sd = sqrt(within / df_within))
end

"""
	groups(data)

The responses of a NIST file (second column), split by group (first column).
"""
groups(data) = [data[data[:, 1] .== g, 2] for g in unique(data[:, 1])]

"""
	anova(data)

The ANOVA table of a NIST file by `OneWayANOVATest`.
"""
function anova(data)
	test = OneWayANOVATest(groups(data)...)
	return anova_table(sum(test.SStᵢ), sum(test.SSeᵢ), test.DFt, test.DFe)
end

"""
	exact_anova(data)

The ANOVA table of a NIST file by the textbook formulas in `BigFloat`: the data limit.
"""
function exact_anova(data)
	g = [big.(x) for x in groups(data)]
	mean_of(x) = sum(x) / length(x)
	grand = mean_of(reduce(vcat, g))
	between = sum(length(x) * (mean_of(x) - grand)^2 for x in g)
	within = sum(sum((x .- mean_of(x)) .^ 2) for x in g)
	return anova_table(between, within, length(g) - 1, sum(length, g) - length(g))
end

columns = ["Julia", "data limit"]
results = []
for (name, level) in datasets
	(; data, header) = read_strd(name)
	b = parse.(Float64, match(r"Between.*? \d+ +(\S+) +(\S+) +(\S+)", header).captures)  # SS, MS, F
	w = parse.(Float64, match(r"Within.*? \d+ +(\S+) +(\S+)", header).captures)  # SS, MS
	nist = [("between SS", :between, b[1]), ("between MS", :between_mean, b[2]), ("F", :F, b[3]),
		("within SS", :within, w[1]), ("within MS", :within_mean, w[2]),
		("R-squared", :r2, certified(header, r"R-Squared +(?<v>\S+)")),
		("residual SD", :sd, certified(header, r"Standard Deviation +(?<v>\S+)"))]
	julia, exact = anova(data), exact_anova(data)
	fewest = report("$name ($level difficulty, $(size(data, 1)) observations)", columns,
		[(label, c, getproperty(julia, key), getproperty(exact, key)) for (label, key, c) in nist])
	push!(results, (name, level, fewest))
end

## 3. Summary

The fewest significant digits shared with NIST in each dataset.

In [ ]:
summary(columns, results)

## What the summary shows

Julia gives 12 to 14 digits on the four lower-difficulty datasets, and AtmWtAg's 10, which is its data limit. From SmLs04 on, the values have 7 and then 13 leading digits that never change, and digits are lost twice. The first loss is the storing: `1000000000000.4` is stored with an error of 0.0000244, which is large beside the spread of 0.1 to 0.5 behind it, and in SmLs07 to SmLs09 the data limit itself drops to 3 or 4 digits. The second is the method: `OneWayANOVATest` gives 1 or 2 digits fewer than the data limit in each of SmLs04 to SmLs09. Subtracting one value of the data from all of them, which leaves every sum of squares unchanged, recovers those digits: in a check on SmLs04, SmLs07 and SmLs09, the between-groups sum of squares then reached the data limit.